In [1]:
import os, json, shutil
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA RTX A4000


In [2]:
ROOT = os.path.abspath(os.path.join('..', '..'))
BASE = os.path.join(ROOT, 'Roman', 'Sentiment Analysis')

dfa_train = pd.read_csv(os.path.join(BASE, 'Domain_A_Mixed_Public_Discourse', 'train.csv'), encoding='utf-8-sig')
dfa_test  = pd.read_csv(os.path.join(BASE, 'Domain_A_Mixed_Public_Discourse', 'test.csv'),  encoding='utf-8-sig')
dfb_train = pd.read_csv(os.path.join(BASE, 'Domain_B_Social_Media_Politics_Drama', 'train.csv'), encoding='utf-8-sig')
dfb_test  = pd.read_csv(os.path.join(BASE, 'Domain_B_Social_Media_Politics_Drama', 'test.csv'),  encoding='utf-8-sig')
dfc_train = pd.read_csv(os.path.join(BASE, 'Domain_C_YouTube_Entertainment', 'train.csv'), encoding='utf-8-sig')
dfc_test  = pd.read_csv(os.path.join(BASE, 'Domain_C_YouTube_Entertainment', 'test.csv'),  encoding='utf-8-sig')
dfd_train = pd.read_csv(os.path.join(BASE, 'Domain_D_Utilities_Apps_Courses', 'train.csv'), encoding='utf-8-sig')
dfd_test  = pd.read_csv(os.path.join(BASE, 'Domain_D_Utilities_Apps_Courses', 'test.csv'),  encoding='utf-8-sig')

# Labels are already numeric: 0=Negative, 1=Positive
LABEL_MAP = {0: 0, 1: 1}
ID2LABEL  = {0: 'NEGATIVE', 1: 'POSITIVE'}
LABEL2ID  = {'NEGATIVE': 0, 'POSITIVE': 1}

print('Domain sizes (train / test):')
for name, tr, te in [
    ('A_Mixed_Public_Discourse',        dfa_train, dfa_test),
    ('B_Social_Media_Politics_Drama',   dfb_train, dfb_test),
    ('C_YouTube_Entertainment',         dfc_train, dfc_test),
    ('D_Utilities_Apps_Courses',        dfd_train, dfd_test),
]:
    print(f'  {name}: train={len(tr)}  test={len(te)}  labels={tr["label"].value_counts().to_dict()}')

Domain sizes (train / test):
  A_Mixed_Public_Discourse: train=19295  test=2156  labels={1: 10320, 0: 8975}
  B_Social_Media_Politics_Drama: train=8799  test=2200  labels={1: 4566, 0: 4233}
  C_YouTube_Entertainment: train=16266  test=4067  labels={1: 9505, 0: 6761}
  D_Utilities_Apps_Courses: train=8799  test=2200  labels={1: 4566, 0: 4233}


In [3]:
XLM_MODEL_ID  = 'xlm-roberta-base'
BERT_MODEL_ID = 'bert-base-multilingual-cased'

NUM_LABELS    = 2
MAX_LEN       = 128
MAX_TRAIN     = 8000
EPOCHS        = 5
PATIENCE      = 2
BATCH_TRAIN   = 16
BATCH_EVAL    = 32
LR            = 2e-5

RESULTS_BASE  = os.path.join(ROOT, 'results', 'T7_Roman_Sentiments')
os.makedirs(RESULTS_BASE, exist_ok=True)

domains = [
    ('A_Mixed_Public_Discourse',       dfa_train, dfa_test),
    ('B_Social_Media_Politics_Drama',  dfb_train, dfb_test),
    ('C_YouTube_Entertainment',        dfc_train, dfc_test),
    ('D_Utilities_Apps_Courses',       dfd_train, dfd_test),
]

In [4]:
def cap_dataset(df, max_samples=MAX_TRAIN):
    if len(df) <= max_samples:
        return df
    samples = []
    for label, group in df.groupby('label'):
        n = min(len(group), round(max_samples * len(group) / len(df)))
        samples.append(group.sample(n=n, random_state=42))
    capped = pd.concat(samples, ignore_index=True)
    return capped.sample(frac=1, random_state=42).reset_index(drop=True)


class UrduDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.labels = df['label'].astype(int).tolist()
        self.enc = tokenizer(
            df['text'].astype(str).tolist(),
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds)
    }


def run_one(model_id, model_label, src_name, train_df, tgt_name, test_df):
    out_dir     = os.path.join(RESULTS_BASE, model_label)
    os.makedirs(out_dir, exist_ok=True)
    result_path = os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')

    if os.path.exists(result_path):
        print(f'  [SKIP] {src_name} -> {tgt_name} already done.')
        with open(result_path, encoding='utf-8') as f:
            return json.load(f)['macro_f1'], None

    run_type = 'IN-DOMAIN' if src_name == tgt_name else 'CROSS-DOMAIN'
    capped   = cap_dataset(train_df)
    print(f'\n  [{run_type}] Train: {src_name} ({len(capped)}) -> Test: {tgt_name} ({len(test_df)})')

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS)

    val_df    = capped.sample(max(int(len(capped) * 0.1), 1), random_state=42)
    train_sub = capped.drop(val_df.index)

    ckpt_dir = os.path.join(out_dir, f'_ckpt_{src_name}_{tgt_name}')

    args = TrainingArguments(
        output_dir                  = ckpt_dir,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_TRAIN,
        per_device_eval_batch_size  = BATCH_EVAL,
        learning_rate               = LR,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        logging_steps               = 50,
        fp16                        = torch.cuda.is_available(),
        report_to                   = 'none',
        save_total_limit            = 1,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = UrduDataset(train_sub, tokenizer),
        eval_dataset    = UrduDataset(val_df, tokenizer),
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    )
    trainer.train()

    preds_out = trainer.predict(UrduDataset(test_df, tokenizer))
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = preds_out.label_ids

    macro_f1 = f1_score(labels, preds, average='macro')
    accuracy = accuracy_score(labels, preds)

    result = {
        'task': 'T7_Roman_Sentiments', 'model': model_label, 'model_id': model_id,
        'source': src_name, 'target': tgt_name, 'type': run_type,
        'train_size': len(capped), 'test_size': len(test_df),
        'macro_f1': round(macro_f1, 4), 'accuracy': round(accuracy, 4),
        'classification_report': classification_report(labels, preds, output_dict=True)
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    if os.path.exists(ckpt_dir):
        shutil.rmtree(ckpt_dir)

    print(f'  macro-F1={macro_f1:.4f}  accuracy={accuracy:.4f}')
    return macro_f1, trainer


print('Helpers loaded. Ready to run experiments.')

Helpers loaded. Ready to run experiments.


## Model 1 — XLM-R Base

In [5]:
print('=== XLM-R | Source: A_Mixed_Public_Discourse ===')
xlmr_results = globals().get('xlmr_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

=== XLM-R | Source: A_Mixed_Public_Discourse ===
  [SKIP] A_Mixed_Public_Discourse -> A_Mixed_Public_Discourse already done.
  [SKIP] A_Mixed_Public_Discourse -> B_Social_Media_Politics_Drama already done.
  [SKIP] A_Mixed_Public_Discourse -> C_YouTube_Entertainment already done.
  [SKIP] A_Mixed_Public_Discourse -> D_Utilities_Apps_Courses already done.

Source A done.


In [6]:
print('=== XLM-R | Source: B_Social_Media_Politics_Drama ===')

src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

=== XLM-R | Source: B_Social_Media_Politics_Drama ===
  [SKIP] B_Social_Media_Politics_Drama -> A_Mixed_Public_Discourse already done.
  [SKIP] B_Social_Media_Politics_Drama -> B_Social_Media_Politics_Drama already done.
  [SKIP] B_Social_Media_Politics_Drama -> C_YouTube_Entertainment already done.
  [SKIP] B_Social_Media_Politics_Drama -> D_Utilities_Apps_Courses already done.

Source B done.


In [7]:
print('=== XLM-R | Source: C_YouTube_Entertainment ===')

src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource C done.')

=== XLM-R | Source: C_YouTube_Entertainment ===
  [SKIP] C_YouTube_Entertainment -> A_Mixed_Public_Discourse already done.
  [SKIP] C_YouTube_Entertainment -> B_Social_Media_Politics_Drama already done.
  [SKIP] C_YouTube_Entertainment -> C_YouTube_Entertainment already done.
  [SKIP] C_YouTube_Entertainment -> D_Utilities_Apps_Courses already done.

Source C done.


In [8]:
print('=== XLM-R | Source: D_Utilities_Apps_Courses ===')

src_name, train_df, _ = domains[3]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nXLM-R -- all 16 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(xlmr_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')

=== XLM-R | Source: D_Utilities_Apps_Courses ===
  [SKIP] D_Utilities_Apps_Courses -> A_Mixed_Public_Discourse already done.
  [SKIP] D_Utilities_Apps_Courses -> B_Social_Media_Politics_Drama already done.
  [SKIP] D_Utilities_Apps_Courses -> C_YouTube_Entertainment already done.
  [SKIP] D_Utilities_Apps_Courses -> D_Utilities_Apps_Courses already done.

XLM-R -- all 16 runs complete.
Results so far:
  [IN ] A_Mixed_Public_Discourse -> A_Mixed_Public_Discourse: 0.8198
  [X  ] A_Mixed_Public_Discourse -> B_Social_Media_Politics_Drama: 0.5768
  [X  ] A_Mixed_Public_Discourse -> C_YouTube_Entertainment: 0.7346
  [X  ] A_Mixed_Public_Discourse -> D_Utilities_Apps_Courses: 0.5768
  [X  ] B_Social_Media_Politics_Drama -> A_Mixed_Public_Discourse: 0.7486
  [IN ] B_Social_Media_Politics_Drama -> B_Social_Media_Politics_Drama: 0.8330
  [X  ] B_Social_Media_Politics_Drama -> C_YouTube_Entertainment: 0.7663
  [X  ] B_Social_Media_Politics_Drama -> D_Utilities_Apps_Courses: 0.8329
  [X  ] C_YouTu

## Model 2 — mBERT

In [9]:
print('=== mBERT | Source: A_Mixed_Public_Discourse ===')
mbert_results = globals().get('mbert_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

=== mBERT | Source: A_Mixed_Public_Discourse ===
  [SKIP] A_Mixed_Public_Discourse -> A_Mixed_Public_Discourse already done.
  [SKIP] A_Mixed_Public_Discourse -> B_Social_Media_Politics_Drama already done.
  [SKIP] A_Mixed_Public_Discourse -> C_YouTube_Entertainment already done.
  [SKIP] A_Mixed_Public_Discourse -> D_Utilities_Apps_Courses already done.

Source A done.


In [10]:
print('=== mBERT | Source: B_Social_Media_Politics_Drama ===')

src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

=== mBERT | Source: B_Social_Media_Politics_Drama ===
  [SKIP] B_Social_Media_Politics_Drama -> A_Mixed_Public_Discourse already done.
  [SKIP] B_Social_Media_Politics_Drama -> B_Social_Media_Politics_Drama already done.
  [SKIP] B_Social_Media_Politics_Drama -> C_YouTube_Entertainment already done.
  [SKIP] B_Social_Media_Politics_Drama -> D_Utilities_Apps_Courses already done.

Source B done.


In [11]:
print('=== mBERT | Source: C_YouTube_Entertainment ===')

src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource C done.')

=== mBERT | Source: C_YouTube_Entertainment ===
  [SKIP] C_YouTube_Entertainment -> A_Mixed_Public_Discourse already done.
  [SKIP] C_YouTube_Entertainment -> B_Social_Media_Politics_Drama already done.
  [SKIP] C_YouTube_Entertainment -> C_YouTube_Entertainment already done.
  [SKIP] C_YouTube_Entertainment -> D_Utilities_Apps_Courses already done.

Source C done.


In [12]:
print('=== mBERT | Source: D_Utilities_Apps_Courses ===')

src_name, train_df, _ = domains[3]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nmBERT -- all 16 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(mbert_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')

=== mBERT | Source: D_Utilities_Apps_Courses ===
  [SKIP] D_Utilities_Apps_Courses -> A_Mixed_Public_Discourse already done.
  [SKIP] D_Utilities_Apps_Courses -> B_Social_Media_Politics_Drama already done.
  [SKIP] D_Utilities_Apps_Courses -> C_YouTube_Entertainment already done.
  [SKIP] D_Utilities_Apps_Courses -> D_Utilities_Apps_Courses already done.

mBERT -- all 16 runs complete.
Results so far:
  [IN ] A_Mixed_Public_Discourse -> A_Mixed_Public_Discourse: 0.7960
  [X  ] A_Mixed_Public_Discourse -> B_Social_Media_Politics_Drama: 0.7501
  [X  ] A_Mixed_Public_Discourse -> C_YouTube_Entertainment: 0.7445
  [X  ] A_Mixed_Public_Discourse -> D_Utilities_Apps_Courses: 0.7501
  [X  ] B_Social_Media_Politics_Drama -> A_Mixed_Public_Discourse: 0.6865
  [IN ] B_Social_Media_Politics_Drama -> B_Social_Media_Politics_Drama: 0.8255
  [X  ] B_Social_Media_Politics_Drama -> C_YouTube_Entertainment: 0.7657
  [X  ] B_Social_Media_Politics_Drama -> D_Utilities_Apps_Courses: 0.8136
  [X  ] C_YouTu

## Table 4 — Compute and Save Results

In [13]:
def compute_table4(results, model_label):
    names = [d[0] for d in domains]
    ss = [results[(d, d)] for d in names]
    st = [results[(s, t)] for s in names for t in names if s != t]
    sd_pairs = [results[(s,s)] - results[(s,t)] for s in names for t in names if s!=t]
    td_pairs = [results[(t,t)] - results[(s,t)] for s in names for t in names if s!=t]

    row = {
        'model':  model_label,
        'task':   'T7_Roman_Sentiments',
        'avg_SS': round(np.mean(ss), 4),
        'avg_ST': round(np.mean(st), 4),
        'avg_SD': round(np.mean(sd_pairs), 4),
        'WSD':    round(max(sd_pairs), 4),
        'avg_TD': round(np.mean(td_pairs), 4),
        'WTD':    round(max(td_pairs), 4),
    }
    print(f"\n{model_label}")
    print(f"  Avg In-Domain  (SS) : {row['avg_SS']}")
    print(f"  Avg Cross-Domain(ST): {row['avg_ST']}")
    print(f"  Avg Source Drop (SD): {row['avg_SD']}")
    print(f"  Worst Source Drop   : {row['WSD']}")
    print(f"  Avg Target Drop (TD): {row['avg_TD']}")
    print(f"  Worst Target Drop   : {row['WTD']}")
    return row

row_xlmr  = compute_table4(xlmr_results,  'XLM-R_Base')
row_mbert = compute_table4(mbert_results, 'mBERT')

summary_df = pd.DataFrame([row_xlmr, row_mbert])
print('\n-- Summary --')
print(summary_df.to_string(index=False))


XLM-R_Base
  Avg In-Domain  (SS) : 0.8526
  Avg Cross-Domain(ST): 0.7205
  Avg Source Drop (SD): 0.1321
  Worst Source Drop   : 0.2891
  Avg Target Drop (TD): 0.1321
  Worst Target Drop   : 0.2562

mBERT
  Avg In-Domain  (SS) : 0.8456
  Avg Cross-Domain(ST): 0.7216
  Avg Source Drop (SD): 0.124
  Worst Source Drop   : 0.3229
  Avg Target Drop (TD): 0.124
  Worst Target Drop   : 0.1951

-- Summary --
     model                task  avg_SS  avg_ST  avg_SD    WSD  avg_TD    WTD
XLM-R_Base T7_Roman_Sentiments  0.8526  0.7205  0.1321 0.2891  0.1321 0.2562
     mBERT T7_Roman_Sentiments  0.8456  0.7216  0.1240 0.3229  0.1240 0.1951


In [14]:
out_path = os.path.join(ROOT, 'results', 'Table4_T7_Roman_Sentiments.csv')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
summary_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

Saved: c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\results\Table4_T7_Roman_Sentiments.csv


---
## Table 5 — Few-Shot LLM Experiments (Robustness)

Using 5 HuggingFace models on GPU with 4-bit quantization:
- **Qwen2.5-7B-Instruct** — Best multilingual support including Urdu
- **Llama-3.1-8B-Instruct** — Strong general-purpose multilingual
- **Gemma-2-9B-it** — Google's instruction-tuned model
- **Aya-Expanse-8B** — Cohere's model built for 23 languages incl. Urdu
- **Mistral-Nemo-Instruct-2407** — 12B joint Mistral+NVIDIA multilingual

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import gc
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"


LLM_MODELS = {
    'Qwen2.5-7B':      'Qwen/Qwen2.5-7B-Instruct',
    'Llama3.1-8B':     'meta-llama/Llama-3.1-8B-Instruct',
    'Mistral-7B':      'mistralai/Mistral-7B-Instruct-v0.3',
}

K_SHOT = 5
LLM_RESULTS_BASE = os.path.join(ROOT, 'results', 'T7_Roman_Sentiments')

print('LLM config ready.')
print(f'Models: {list(LLM_MODELS.keys())}')
print(f'K-shot: {K_SHOT} examples per class')
print('Note: Only Mistral-7B and meta-llama-8B included for focused testing')

LLM config ready.
Models: ['Qwen2.5-7B', 'Llama3.1-8B', 'Mistral-7B']
K-shot: 5 examples per class
Note: Only Mistral-7B and meta-llama-8B included for focused testing


In [16]:
MAX_TEST_LLM = 100
LLM_BATCH_SIZE = 16

def build_prompt(train_df, test_text, k=K_SHOT):
    examples = []
    for label_val in sorted(train_df['label'].unique()):
        subset = train_df[train_df['label'] == label_val]
        sampled = subset.sample(n=min(k, len(subset)), random_state=42)
        for _, row in sampled.iterrows():
            label_str = 'Positive' if row['label'] == 1 else 'Negative'
            examples.append(f"Text: {row['text']}\nLabel: {label_str}")

    prompt = (
        "You are a sentiment classifier for Roman Urdu text. "
        "Classify each text as either 'Positive' or 'Negative'. "
        "Respond with ONLY the label, nothing else.\n\n"
        "Examples:\n" + "\n\n".join(examples) + "\n\n"
        f"Text: {test_text}\nLabel:"
    )
    return prompt


def parse_llm_label(response):
    response = response.strip().lower()
    if 'positive' in response:
        return 1
    elif 'negative' in response:
        return 0
    elif '1' in response:
        return 1
    elif '0' in response:
        return 0
    return -1


def run_llm_experiment(model_name, model_id, src_name, train_df, tgt_name, test_df, tokenizer, model):
    out_dir = os.path.join(LLM_RESULTS_BASE, model_name)
    os.makedirs(out_dir, exist_ok=True)
    result_path = os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')

    if os.path.exists(result_path):
        print(f'  [SKIP] {src_name} -> {tgt_name} already done.')
        with open(result_path, encoding='utf-8') as f:
            return json.load(f)['macro_f1']

    run_type = 'IN-DOMAIN' if src_name == tgt_name else 'CROSS-DOMAIN'

    if len(test_df) > MAX_TEST_LLM:
        n_per_class = MAX_TEST_LLM // test_df['label'].nunique()
        test_eval = pd.concat([
            group.sample(n=min(len(group), n_per_class), random_state=42)
            for _, group in test_df.groupby('label')
        ]).reset_index(drop=True)
    else:
        test_eval = test_df

    print(f'  [{run_type}] {src_name} -> {tgt_name} ({len(test_eval)} samples)')

    true_labels = test_eval['label'].astype(int).tolist()
    all_prompts = [build_prompt(train_df, row['text']) for _, row in test_eval.iterrows()]

    preds = []
    for i in range(0, len(all_prompts), LLM_BATCH_SIZE):
        batch_prompts = all_prompts[i:i+LLM_BATCH_SIZE]
        inputs = tokenizer(
            batch_prompts, return_tensors='pt', truncation=True,
            max_length=1024, padding=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=10, do_sample=False,
                temperature=1.0, pad_token_id=tokenizer.eos_token_id
            )

        for j, output in enumerate(outputs):
            input_len = inputs['input_ids'][j].ne(tokenizer.pad_token_id).sum()
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred = parse_llm_label(generated)
            if pred == -1:
                pred = 0
            preds.append(pred)

    macro_f1 = f1_score(true_labels, preds, average='macro')
    accuracy = accuracy_score(true_labels, preds)

    result = {
        'task': 'T7_Roman_Sentiments', 'model': model_name, 'model_id': model_id,
        'source': src_name, 'target': tgt_name, 'type': run_type,
        'k_shot': K_SHOT, 'test_size': len(test_eval),
        'macro_f1': round(macro_f1, 4), 'accuracy': round(accuracy, 4),
        'classification_report': classification_report(true_labels, preds, output_dict=True, zero_division=0)
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f'    F1={macro_f1:.4f}  Acc={accuracy:.4f}')
    return macro_f1


print('LLM helpers loaded (batched inference, max test =', MAX_TEST_LLM, ')')

LLM helpers loaded (batched inference, max test = 100 )


In [17]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

all_llm_results = {}

for model_name, model_id in LLM_MODELS.items():
    out_dir = os.path.join(LLM_RESULTS_BASE, model_name)
    all_done = True
    for src_name, _, _ in domains:
        for tgt_name, _, _ in domains:
            if not os.path.exists(os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')):
                all_done = False
                break
        if not all_done:
            break

    if all_done:
        print(f'\n[SKIP ALL] {model_name} — all 16 results already exist.')
        llm_results = {}
        for src_name, _, _ in domains:
            for tgt_name, _, _ in domains:
                with open(os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json'), encoding='utf-8') as f:
                    llm_results[(src_name, tgt_name)] = json.load(f)['macro_f1']
        all_llm_results[model_name] = llm_results
        continue

    print(f'\n{"="*60}')
    print(f'Loading {model_name} ({model_id})...')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map='auto',
        token=HF_TOKEN,
        disable_mmap=True
    )
    model.eval()
    print(f'{model_name} loaded successfully.')

    llm_results = {}
    for src_name, train_df, _ in domains:
        for tgt_name, _, test_df in domains:
            f1 = run_llm_experiment(model_name, model_id, src_name, train_df, tgt_name, test_df, tokenizer, model)
            llm_results[(src_name, tgt_name)] = f1

    all_llm_results[model_name] = llm_results

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\n{model_name} -- all 16 runs complete. Memory freed.')


[SKIP ALL] Qwen2.5-7B — all 16 results already exist.

[SKIP ALL] Llama3.1-8B — all 16 results already exist.

Loading Mistral-7B (mistralai/Mistral-7B-Instruct-v0.3)...


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maryam\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 166.26it/s

Mistral-7B loaded successfully.
  [IN-DOMAIN] A_Mixed_Public_Discourse -> A_Mixed_Public_Discourse (100 samples)
    F1=0.6484  Acc=0.6800
  [CROSS-DOMAIN] A_Mixed_Public_Discourse -> B_Social_Media_Politics_Drama (100 samples)
    F1=0.4579  Acc=0.5100
  [CROSS-DOMAIN] A_Mixed_Public_Discourse -> C_YouTube_Entertainment (100 samples)
    F1=0.5175  Acc=0.5800
  [CROSS-DOMAIN] A_Mixed_Public_Discourse -> D_Utilities_Apps_Courses (100 samples)
    F1=0.4579  Acc=0.5100
  [CROSS-DOMAIN] B_Social_Media_Politics_Drama -> A_Mixed_Public_Discourse (100 samples)
    F1=0.6264  Acc=0.6600
  [IN-DOMAIN] B_Social_Media_Politics_Drama -> B_Social_Media_Politics_Drama (100 samples)
    F1=0.4726  Acc=0.5300
  [CROSS-DOMAIN] B_Social_Media_Politics_Drama -> C_YouTube_Entertainment (100 samples)
    F1=0.5703  Acc=0.6200
  [CROSS-DOMAIN] B_Social_Media_Politics_Drama -> D_Utilities_Apps_Courses (100 samples)
    F1=0.4726  Acc=0.5300
  [CROSS-DOMAIN] C_YouTube_Entertainment -> A_Mixed_Public_Discour

## Table 5 — Compute and Save Results

In [18]:
def compute_table5_row(results, model_label):
    names = [d[0] for d in domains]
    st = [results[(s, t)] for s in names for t in names if s != t]
    ss = [results[(d, d)] for d in names]
    return {
        'model': model_label,
        'avg_SS': round(np.mean(ss), 4),
        'avg_ST': round(np.mean(st), 4),
    }

table5_rows = []
table5_rows.append({**compute_table5_row(xlmr_results, 'XLM-R_Base (fine-tuned)'), 'type': 'fine-tuned'})
table5_rows.append({**compute_table5_row(mbert_results, 'mBERT (fine-tuned)'), 'type': 'fine-tuned'})

for model_name, results in all_llm_results.items():
    table5_rows.append({**compute_table5_row(results, f'{model_name} (5-shot)'), 'type': 'few-shot'})

table5_df = pd.DataFrame(table5_rows)
print('\n=== TABLE 5: Robustness Comparison (Roman Sentiments) ===')
print(table5_df.to_string(index=False))

out_path = os.path.join(ROOT, 'results', 'Table5_T7_Roman_Sentiments.csv')
table5_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')


=== TABLE 5: Robustness Comparison (Roman Sentiments) ===
                  model  avg_SS  avg_ST       type
XLM-R_Base (fine-tuned)  0.8526  0.7205 fine-tuned
     mBERT (fine-tuned)  0.8456  0.7216 fine-tuned
    Qwen2.5-7B (5-shot)  0.5096  0.5133   few-shot
   Llama3.1-8B (5-shot)  0.4886  0.4504   few-shot
    Mistral-7B (5-shot)  0.5466  0.5292   few-shot

Saved: c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\results\Table5_T7_Roman_Sentiments.csv


---
## HuggingFace Upload — Open Source Contribution

In [21]:
names = [d[0] for d in domains]

ss_scores_mbert = {name: mbert_results[(name, name)] for name in names}
best_domain_mbert = max(ss_scores_mbert, key=ss_scores_mbert.get)
best_domain_mbert_df = next(tr for n, tr, _ in domains if n == best_domain_mbert)
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

Logged in to HuggingFace.


In [22]:
ss_scores_mbert = {name: mbert_results[(name, name)] for name in names}
best_domain_mbert = max(ss_scores_mbert, key=ss_scores_mbert.get)
best_domain_mbert_df = next(tr for n, tr, _ in domains if n == best_domain_mbert)

print(f'Best in-domain (mBERT): {best_domain_mbert}  (F1={ss_scores_mbert[best_domain_mbert]:.4f})')
print(f'Re-training on full {len(best_domain_mbert_df)} samples for HuggingFace upload...')

REPO_MBERT = f'{HF_USERNAME}/mbert-roman-urdu-sentiment'
HF_CKPT_MBERT = os.path.join(RESULTS_BASE, 'hf_upload_mbert')

tokenizer_mbert = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
model_mbert = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_ID,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

val_mbert   = best_domain_mbert_df.sample(max(int(len(best_domain_mbert_df) * 0.1), 1), random_state=42)
train_mbert = best_domain_mbert_df.drop(val_mbert.index)

mbert_args = TrainingArguments(
    output_dir                  = HF_CKPT_MBERT,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_TRAIN,
    per_device_eval_batch_size  = BATCH_EVAL,
    learning_rate               = LR,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    save_total_limit            = 1,
    push_to_hub                 = True,
    hub_model_id                = REPO_MBERT,
    hub_token                   = HF_TOKEN,
)

mbert_trainer = Trainer(
    model           = model_mbert,
    args            = mbert_args,
    train_dataset   = UrduDataset(train_mbert, tokenizer_mbert),
    eval_dataset    = UrduDataset(val_mbert,   tokenizer_mbert),
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
)

mbert_trainer.train()

mbert_trainer.push_to_hub(
    commit_message='Upload mBERT fine-tuned on Roman Urdu Sentiment Analysis'
)
tokenizer_mbert.push_to_hub(REPO_MBERT, token=HF_TOKEN)

print(f'\nmBERT model pushed to: https://huggingface.co/{REPO_MBERT}')

if os.path.exists(HF_CKPT_MBERT):
    shutil.rmtree(HF_CKPT_MBERT)

Best in-domain (mBERT): C_YouTube_Entertainment  (F1=0.9344)
Re-training on full 16266 samples for HuggingFace upload...


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maryam\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14211.69it/s]
[tr

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.367697,0.204840,0.922684,0.924354
2,0.175693,0.225705,0.933101,0.935424
3,0.095751,0.305886,0.935649,0.937269
4,0.063391,0.336360,0.937747,0.939114
5,0.033917,0.363923,0.943677,0.945264


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.88it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber


mBERT model pushed to: https://huggingface.co/abdullaharoon/mbert-roman-urdu-sentiment
